# 03 特征筛选：任一指标有效，短期与长期共同保留

使用内置 **hscredit dataset 的全部 X**，以 `MOB1@30 = MOB1 > 30`、`MOB6@30 = MOB6 > 30` 为标签。
观察时长严格要求 `etl_date > 放款时间 + 6 个日历月 + 30 天`，**不限制借款期限**。

本例的判定层次：

1. 同一个样本集、同一个标签：**IV / KS / AUC / 头部 Lift 任一达标即有效（OR）**。
2. 同一个字段：短期与长期、训练与验证这四组都要各有至少一个指标达标（AND）；不同组可以由不同指标命中。
3. 质量门槛、PSI 与相关性去冗余仍分别执行，不能由有效性指标替代。
4. 时间外复核沿用同样的“每标签任一指标达标”规则，单独报告，不回写开发阶段的字段集合。

代码先按“上一步输出 → 当前筛选器 → 下一步输入”逐步执行，最后使用 **sklearn Pipeline 串联同样步骤并断言结果一致**。
类型分支与有效性指标属于并列分支，使用并集组合；后面另外演示交集组合、评分卡组合、嵌入法和包装法。

数据口径见 [内置数据集说明](../hscredit/resources/datasets/README.md)。全量加载约 1 GiB，筛选与 Pipeline 复核需要更多内存。
这是一组经验候选规则；PSI 稳定并不意味着效果稳定。IV、KS 不辨方向，因此还会展示方向变化提示，提示不额外阻断“任一有效”的入选规则。

In [1]:
from pathlib import Path
import sys

# 同时支持从仓库根目录或 examples/ 启动；安装后的 hscredit 也可直接导入。
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hscredit" / "__init__.py").exists()), None)
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import hscredit
from hscredit import hscredit_dataset
from hscredit.core.binning import OptimalBinning
from hscredit.core.selectors import (
    NullSelector, ModeSelector, TypeSelector, RegexSelector, CardinalitySelector,
    VarianceSelector, IVSelector, KSSelector, LiftSelector, PSISelector,
    CorrSelector, VIFSelector, FeatureImportanceSelector, NullImportanceSelector,
    MutualInfoSelector, Chi2Selector, FTestSelector, RFESelector,
    SequentialFeatureSelector, StepwiseSelector, BorutaSelector,
    StabilityAwareSelector, CompositeFeatureSelector, ScorecardFeatureSelection,
    BaseFeatureSelector, SelectionReportCollector, collect_selection_report,
)

SEED = 42
N_JOBS = 2  # 有界并行；可按机器资源调整，模型内部保持 n_jobs=1。
LABELS = ["MOB1@30", "MOB6@30"]
LIMITS = {"缺失率": 0.98, "单一值率": 0.98, "类别基数": 50,
          "IV": 0.02, "KS": 0.05, "AUC": 0.53, "头部Lift": 1.20,
          "PSI": 0.25, "相关性": 0.80}
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 16)
display(pd.Series(LIMITS, name="演示门槛").to_frame())

BIN_PARAMS = {"method": "quantile", "max_n_bins": 5, "min_n_bins": 1,
              "min_bin_size": 0.01, "missing_separate": True}
METRIC_LIMITS = {"IV": LIMITS["IV"], "KS": LIMITS["KS"],
                 "AUC": LIMITS["AUC"], "Lift": LIMITS["头部Lift"]}

,演示门槛
缺失率,0.9800
单一值率,0.9800
类别基数,50.0000
IV,0.0200
KS,0.0500
AUC,0.5300
头部Lift,1.2000
PSI,0.2500
相关性,0.8000


### 两个 Notebook 内的轻量扩展

当前内置 `CompositeFeatureSelector` 支持 `sequential`、`intersection`。这里用公共基类实现 `UnionFeatureSelector`，
让并列分支共享同一输入、按字段名求并集，保持原始列顺序，并保留每个子筛选器的报告。
`AUCSelector` 则补齐有方向的 AUC 门槛。它们仅定义在本 Notebook，不修改库的公共接口。

In [2]:
class AUCSelector(BaseFeatureSelector):
    """按有方向的 AUC 筛选，沿用基类接口。

    **参数**
    threshold：AUC 门槛；其余参数继承 BaseFeatureSelector。

    **属性**
    scores_：各字段 AUC；selected_features_：入选字段。

    **参考样例**
    AUCSelector(threshold=0.53).fit(X, y).transform(X)
    """

    method_name = "AUC筛选"

    def _fit_impl(self, X, y):
        self.scores_ = pd.Series({f: roc_auc_score(y, X[f]) for f in X})
        self.threshold_ = self.threshold
        self.score_name_, self.score_direction_ = "AUC", "越大越好"
        passed = self.scores_.ge(self.threshold)
        self.selected_features_ = passed.index[passed].tolist()
        self.condition_results_ = passed.to_frame("AUC达标")
        self._drop_reason = f"AUC < {self.threshold}"


class UnionFeatureSelector(BaseFeatureSelector):
    """各子筛选器独立拟合同一输入，再取并集。

    **参数**
    selectors：(名称, 筛选器) 列表；n_jobs：子筛选器的并行预算配置。

    **属性**
    selected_features_：入选字段；scores_：各字段命中的分支数。

    **参考样例**
    UnionFeatureSelector([("IV", IVSelector()), ("KS", KSSelector())]).fit(X, y)
    """

    method_name = "任一分支通过"

    def __init__(self, selectors, n_jobs=1):
        super().__init__(n_jobs=n_jobs, parallel_backend="threading")
        self.selectors = selectors
        self.strategy = "union"

    def _fit_impl(self, X, y):
        fitted, votes = [], pd.DataFrame(index=X.columns)
        for name, selector in self.selectors:
            step = clone(selector)
            if "parallel_backend" in step.get_params(deep=False):
                step.set_params(parallel_backend="threading")
            step.fit(X, y)
            fitted.append((name, step))
            votes[name] = votes.index.isin(step.selected_features_)
        self.selectors = fitted
        self.executed_stages_ = list(range(len(fitted)))
        self.skipped_stages_ = {}
        self.scores_ = votes.sum(axis=1)
        self.score_name_, self.score_direction_ = "命中分支数", "越大越好"
        self.threshold_ = 1
        self.selected_features_ = votes.index[votes.any(axis=1)].tolist()
        self.selection_reasons_ = votes.apply(lambda row: "命中：" + "、".join(row.index[row]) if row.any() else "所有分支均未通过", axis=1)

## 1. 数据、共同样本与标签

`hscredit_dataset(split=True)` 返回订单信息、贷后表现、授信时点 X。只对 **X** 做筛选；编号、时间、额度、期限、渠道和所有 MOB / DPD 均不进入特征。

共同样本要求：放款时间和抽数日期有效，且 **抽数日期严格晚于放款时间 + 6 个日历月 + 30 天**；不限制 `periods`。
MOB1、MOB6 均非缺失且非负才构造标签，避免将未知表现当成好样本。
用户编码用于隔离客户：同一客户仅保留共同样本中授信最早的一笔，后续切分不会有客户穿越。

In [3]:
order, overdue, X = hscredit_dataset(split=True)
assert order.index.equals(overdue.index) and order.index.equals(X.index)
assert not (set(X.columns) & (set(order.columns) | set(overdue.columns)))

loan_time = pd.to_datetime(order["放款时间"], errors="coerce")
etl_time = pd.to_datetime(order["etl_date"], errors="coerce")
credit_time = pd.to_datetime(order["授信时间"], errors="coerce")
mature_after = loan_time + pd.DateOffset(months=6) + pd.Timedelta(days=30)
mature = loan_time.notna() & etl_time.notna() & etl_time.gt(mature_after)
valid_labels = overdue[["MOB1", "MOB6"]].notna().all(axis=1) & overdue[["MOB1", "MOB6"]].ge(0).all(axis=1)
eligible = mature & valid_labels & credit_time.notna() & order["用户编码"].notna()
eligible_order = order.loc[eligible].assign(授信时间=credit_time.loc[eligible]).sort_values("授信时间", kind="stable")
sample_index = eligible_order.drop_duplicates("用户编码", keep="first").index

sample_order = eligible_order.loc[sample_index].copy()
Y = overdue.loc[sample_index, ["MOB1", "MOB6"]].gt(30).astype("int8")
Y.columns = LABELS
X = X.loc[sample_index].replace([np.inf, -np.inf], np.nan)
# 布尔值转为等价的 0/1，便于方差与数值分箱；不改变字段名。
X = X.astype({f: "float64" for f in X.select_dtypes(include="bool")})
assert X.index.equals(Y.index) and X.index.equals(sample_order.index)
assert sample_order["用户编码"].is_unique
assert Y[LABELS[0]].le(Y[LABELS[1]]).all(), "累计 MOB 标签出现短期坏、长期好，请核查原始口径"

display(pd.DataFrame({"步骤": ["原始订单", "满足观察时长", "标签与关联信息有效", "每客户保留首笔"],
                      "样本数": [len(order), int(mature.sum()), int(eligible.sum()), len(X)]}))
display(pd.DataFrame({"字段组": ["订单信息", "贷后表现", "候选X"], "字段数": [order.shape[1], overdue.shape[1], X.shape[1]]}))
display(Y.agg(["count", "sum", "mean"]).rename(index={"count": "样本数", "sum": "坏样本数", "mean": "坏样本率"}))
# 后续仅需共同样本及订单元信息，释放大表引用。
del order, overdue, eligible_order

,步骤,样本数
0,原始订单,20092
1,满足观察时长,20092
2,标签与关联信息有效,20092
3,每客户保留首笔,20027


,字段组,字段数
0,订单信息,9
1,贷后表现,14
2,候选X,6781


,MOB1@30,MOB6@30
样本数,20027.0000,20027.0000
坏样本数,325.0000,1510.0000
坏样本率,0.0162,0.0754


## 2. 按授信日期划分训练 / 验证 / 时间外样本

按时间顺序约 60% / 20% / 20% 切分，同一天不跨集合。训练集用于拟合分箱、风险方向及模型；
验证集用于双标签有效性和 PSI 筛选，因此它不是最终测试集；最后 20% 的时间外样本留到字段冻结后再评估。
短期、长期始终使用完全相同的行。所有数据清洗参数和分箱规则只在训练集上学习。

In [4]:
credit_day = sample_order["授信时间"].dt.normalize()
valid_start = credit_day.iloc[int(len(credit_day) * 0.60)]
oot_start = credit_day.iloc[int(len(credit_day) * 0.80)]
split_index = {
    "训练": X.index[credit_day.lt(valid_start)],
    "验证": X.index[credit_day.ge(valid_start) & credit_day.lt(oot_start)],
    "时间外": X.index[credit_day.ge(oot_start)],
}
X_train, X_valid = (X.loc[split_index[s]].copy() for s in ["训练", "验证"])
y_train, y_valid = (Y.loc[split_index[s]].copy() for s in ["训练", "验证"])
assert all(len(idx) > 0 and Y.loc[idx].nunique().eq(2).all() for idx in split_index.values())
assert max(sample_order.loc[split_index["训练"], "授信时间"]) < min(sample_order.loc[split_index["验证"], "授信时间"])
assert max(sample_order.loc[split_index["验证"], "授信时间"]) < min(sample_order.loc[split_index["时间外"], "授信时间"])
groups = [set(sample_order.loc[idx, "用户编码"]) for idx in split_index.values()]
assert all(groups[i].isdisjoint(groups[j]) for i in range(3) for j in range(i + 1, 3))
display(pd.DataFrame([
    {"样本集": name, "样本数": len(idx), "最早授信": credit_day.loc[idx].min(), "最晚授信": credit_day.loc[idx].max(),
     **{label + "坏样本率": Y.loc[idx, label].mean() for label in LABELS}}
    for name, idx in split_index.items()
]))

,样本集,样本数,最早授信,最晚授信,MOB1@30坏样本率,MOB6@30坏样本率
0,训练,11973,2025-01-05,2025-03-01,0.0184,0.0733
1,验证,3796,2025-03-02,2025-03-12,0.0124,0.0688
2,时间外,4258,2025-03-13,2025-03-24,0.0136,0.0871


## 3. 基础筛选：每一步接收上一步的输出

先执行缺失率，再执行单一值率。两个筛选器互相独立，均只在训练样本拟合；验证数据只调用 `transform`。
不预先 `dropna()`，不将标签或订单信息混入 X。`get_support()`、`selected_features_`、`report()` 均可直接使用。

In [5]:
# 第一步：缺失率。
null_selector = NullSelector(threshold=LIMITS["缺失率"], n_jobs=N_JOBS, parallel_backend="threading")
X_after_null = null_selector.fit_transform(X_train, y_train[LABELS[0]])
X_valid_after_null = null_selector.transform(X_valid)

# 第二步：单一值率，只接收第一步输出。
mode_selector = ModeSelector(threshold=LIMITS["单一值率"], dropna=False, n_jobs=N_JOBS, parallel_backend="threading")
X_after_mode = mode_selector.fit_transform(X_after_null, y_train[LABELS[0]])
X_valid_after_mode = mode_selector.transform(X_valid_after_null)

assert list(X_after_null) == list(X_train.columns[null_selector.get_support()])
display(null_selector.report()[0])
display(mode_selector.report()[0])

,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,NullSelector,NullSelector,single,已完成,6781,6646,135,0.9801,0.5224,筛选完成


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,ModeSelector,ModeSelector,single,已完成,6646,4742,1904,0.7135,2.6682,筛选完成


### 第三步：类型分支组合

数值字段走 `TypeSelector → VarianceSelector`，类别字段走 `TypeSelector → CardinalitySelector`，最后取两条路径的并集。
基数门槛只作用于类别字段，不误删连续变量。数值方差门槛为 0，仅去除常量，不用统一非零门槛比较不同量纲。
`RegexSelector` 另外展示字段族探索，不作为有效性的判据。

In [6]:
type_selector = UnionFeatureSelector([
    ("数值分支", CompositeFeatureSelector([
        ("数值类型", TypeSelector(dtype_include="number", n_jobs=N_JOBS)),
        ("去除常量", VarianceSelector(threshold=0.0, n_jobs=N_JOBS)),
    ], strategy="sequential", n_jobs=N_JOBS, parallel_backend="threading")),
    ("类别分支", CompositeFeatureSelector([
        ("类别类型", TypeSelector(dtype_exclude="number", n_jobs=N_JOBS)),
        ("类别基数", CardinalitySelector(threshold=LIMITS["类别基数"], n_jobs=N_JOBS)),
    ], strategy="sequential", n_jobs=N_JOBS, parallel_backend="threading")),
], n_jobs=N_JOBS)

X_after_type = type_selector.fit_transform(X_after_mode, y_train[LABELS[0]])
X_valid_after_type = type_selector.transform(X_valid_after_mode)
quality_features = list(X_after_type)
assert list(X_valid_after_type) == quality_features
display(type_selector.report()[0])

family_selector = RegexSelector(pattern=r"^(DJK_|DK_|LZ_)", n_jobs=N_JOBS).fit(X_after_type)
display(pd.DataFrame({"字段族示例": family_selector.selected_features_[:10]}))

,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,UnionFeatureSelector,UnionFeatureSelector,union,已完成,4742,4248,494,0.8958,4.5678,筛选完成
1,2,selector/01_数值分支,selector,数值分支,CompositeFeatureSelector,sequential,已完成,4742,4238,504,0.8937,2.1264,筛选完成
2,3,selector/01_数值分支/01_数值类型,selector/01_数值分支,数值类型,TypeSelector,single,已完成,4742,4732,10,0.9979,0.4008,筛选完成
3,4,selector/01_数值分支/02_去除常量,selector/01_数值分支,去除常量,VarianceSelector,single,已完成,4732,4238,494,0.8956,0.7063,筛选完成
4,5,selector/02_类别分支,selector,类别分支,CompositeFeatureSelector,sequential,已完成,4742,10,4732,0.0021,0.4403,筛选完成
5,6,selector/02_类别分支/01_类别类型,selector/02_类别分支,类别类型,TypeSelector,single,已完成,4742,10,4732,0.0021,0.0853,筛选完成
6,7,selector/02_类别分支/02_类别基数,selector/02_类别分支,类别基数,CardinalitySelector,single,已完成,10,10,0,1.0000,0.0081,筛选完成


,字段族示例
0,DJK_X986
1,DJK_X1546
2,DJK_X1521
3,DJK_X1497
4,DJK_X536
5,DJK_X156
6,DJK_X1342
7,DJK_X1066
8,DJK_X797
9,DJK_X893


## 4. 第四步：双标签任一指标有效

这个步骤只接收第三步输出及对应标签、验证样本，返回有效字段构成的 X。内部重复工作封装成下方可复用组件，主线只需一次 `fit_transform`。

- 所有质量合格字段都进入评估，**没有 IV 必须达标的前置关卡**。
- 分箱只拟合训练集；最多 5 箱，缺失单独成箱，稀疏字段允许合并到一个普通箱。最小箱占比为 1%，与允许最高 98% 缺失的质量门槛匹配，避免把稀疏字段隐式排除。
- 每个箱按训练短期平滑坏率映射为风险分数；验证、长期和时间外均复用同一映射，不反转方向。
- IV 在箱编号上计算；KS / AUC / Lift 在风险分数上独立计算，并由并集组合器聚合。
- 每一组取 `IV通过 OR KS通过 OR AUC通过 OR Lift通过`，再对训练/验证和短期/长期四组取交集。
- `LiftSelector(direction="bad")` 传入的是 `Lift - 1` 的改善门槛，即 `1.20 - 1 = 0.20`。

达标和未达标指标均保留在明细中。AUC < 0.5 仅标记方向反转，不暗中恢复成额外的硬条件。

In [7]:
def evaluate_metrics(bins, risk, target, limits, n_jobs):
    """独立计算四个指标；同一组任一指标通过即入选。"""
    iv = IVSelector(threshold=limits["IV"], n_jobs=n_jobs, parallel_backend="threading").fit(bins, target)
    risk_union = UnionFeatureSelector([
        ("KS", KSSelector(threshold=limits["KS"], n_jobs=n_jobs)),
        ("AUC", AUCSelector(threshold=limits["AUC"], n_jobs=1)),
        ("Lift", LiftSelector(threshold=limits["头部Lift"] - 1, ratio=0.1,
                              direction="bad", tie_policy="fractional", n_jobs=n_jobs)),
    ], n_jobs=n_jobs).fit(risk, target)
    methods = {"IV": iv, **dict(risk_union.selectors)}
    values = pd.DataFrame({name: step.scores_ for name, step in methods.items()})
    values["Lift"] = methods["Lift"].lift_detail_["找坏人LIFT"]
    hits = pd.DataFrame({name: values[name].ge(threshold) for name, threshold in {
        "IV": limits["IV"], "KS": limits["KS"], "AUC": limits["AUC"], "Lift": limits["头部Lift"]}.items()})
    # 用数值判定与筛选器入选名单互相核验，确保并集规则真正生效。
    union_names = set(iv.selected_features_) | set(risk_union.selected_features_)
    assert set(hits.index[hits.any(axis=1)]) == union_names
    return values, hits, iv, risk_union


def map_risk(bins, risk_maps, fallback):
    """只使用训练映射，不读取当前样本标签。"""
    return pd.DataFrame({f: bins[f].map(risk_maps[f]).fillna(fallback).astype(float)
                         for f in bins}, index=bins.index)


class DualPeriodSelector(TransformerMixin, BaseEstimator):
    """封装分箱与双标签任一有效判定，transform 只选择 X 字段。

    **参数**
    limits：筛选阈值；bin_params：分箱参数；n_jobs：并行预算。
    fit 接收短期 y、长期 long_y、验证 X 与双标签 validation_labels。

    **属性**
    selected_features_：入选字段；evidence_：指标证据；metric_hits_：达标矩阵。

    **参考样例**
    selector.fit(X, y, long_y=y6, validation_X=Xv, validation_labels=Yv).transform(X)
    """

    def __init__(self, limits, bin_params, n_jobs=1):
        self.limits = limits
        self.bin_params = bin_params
        self.n_jobs = n_jobs

    def fit(self, X, y, *, long_y, validation_X, validation_labels):
        labels = pd.DataFrame({LABELS[0]: y, LABELS[1]: long_y}, index=X.index)
        assert labels.notna().all().all() and labels.nunique().eq(2).all()
        assert validation_X.index.equals(validation_labels.index)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        self.binner_ = OptimalBinning(**self.bin_params, n_jobs=self.n_jobs,
                                      parallel_backend="threading", random_state=SEED).fit(X, y)
        self.bins_train_ = self.binner_.transform(X)
        self.bins_valid_ = self.binner_.transform(validation_X.loc[:, X.columns])
        self.base_risk_ = float(labels[LABELS[0]].mean())
        self.risk_maps_ = {}
        for feature in X:
            groups = pd.DataFrame({"箱": self.bins_train_[feature], "标签": labels[LABELS[0]]}).groupby("箱", observed=True)["标签"].agg(["sum", "count"])
            self.risk_maps_[feature] = (groups["sum"] + 10 * self.base_risk_) / (groups["count"] + 10)
        self.risk_train_ = map_risk(self.bins_train_, self.risk_maps_, self.base_risk_)
        self.risk_valid_ = map_risk(self.bins_valid_, self.risk_maps_, self.base_risk_)

        self.evidence_ = pd.DataFrame(index=pd.Index(X.columns, name="特征"))
        self.sources_, self.group_pass_, self.metric_hits_ = {}, pd.DataFrame(index=X.columns), {}
        for split, bins, risk, targets in [
            ("训练", self.bins_train_, self.risk_train_, labels),
            ("验证", self.bins_valid_, self.risk_valid_, validation_labels),
        ]:
            for label in LABELS:
                values, hits, iv, risk_union = evaluate_metrics(bins, risk, targets[label], self.limits, self.n_jobs)
                group = f"{split}_{label}"
                self.sources_[group] = (iv, risk_union)
                self.metric_hits_[group] = hits
                self.evidence_ = self.evidence_.join(values.add_prefix(group + "_"))
                self.evidence_[group + "_命中指标"] = hits.apply(lambda row: "、".join(row.index[row]), axis=1)
                self.group_pass_[group] = hits.any(axis=1)
        self.evidence_["双标签有效性通过"] = self.group_pass_.all(axis=1)
        auc_columns = [f"{split}_{label}_AUC" for split in ["训练", "验证"] for label in LABELS]
        self.evidence_["方向反转提示"] = self.evidence_[auc_columns].lt(0.5).any(axis=1)
        self.selected_features_ = self.evidence_.index[self.evidence_["双标签有效性通过"]].tolist()
        return self

    def transform(self, X):
        return X.loc[:, self.selected_features_].copy()

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.selected_features_, dtype=object)

In [8]:
effect_selector = DualPeriodSelector(LIMITS, BIN_PARAMS, n_jobs=N_JOBS)
X_after_effect = effect_selector.fit_transform(
    X_after_type, y_train[LABELS[0]], long_y=y_train[LABELS[1]],
    validation_X=X_valid_after_type, validation_labels=y_valid,
)
X_valid_after_effect = effect_selector.transform(X_valid_after_type)
evidence = effect_selector.evidence_.copy()
performance_features = list(X_after_effect)
assert performance_features, "没有字段满足双标签任一有效规则"

# 命中指标矩阵是判定依据，不先用 IV 缩减字段池。
display(pd.DataFrame({group: hits.sum() for group, hits in effect_selector.metric_hits_.items()}).rename_axis("达标指标"))
display(effect_selector.group_pass_.sum().rename("任一指标通过字段数").to_frame())
display(evidence.loc[performance_features, [c for c in evidence if c.endswith("命中指标")] + ["方向反转提示"]].head(20))
print("训练和验证、短期和长期均至少一个指标有效：", len(performance_features))

,训练_MOB1@30,训练_MOB6@30,验证_MOB1@30,验证_MOB6@30
达标指标,,,,
IV,1476,402,2567,931
KS,1527,612,2281,608
AUC,1489,513,606,174
Lift,1990,684,786,557


,任一指标通过字段数
训练_MOB1@30,2410
训练_MOB6@30,825
验证_MOB1@30,3110
验证_MOB6@30,1101


,训练_MOB1@30_命中指标,训练_MOB6@30_命中指标,验证_MOB1@30_命中指标,验证_MOB6@30_命中指标,方向反转提示
特征,,,,,
province,IV、KS、AUC、Lift,IV、KS、AUC、Lift,IV、KS、Lift,KS,True
DJK_X230,IV、KS、AUC、Lift,KS,IV、KS,IV、KS,True
DJK_X380,Lift,Lift,IV,IV,False
DJK_X345,IV、AUC、Lift,Lift,IV、KS、Lift,IV,True
DJK_X644,IV、AUC、Lift,Lift,IV、KS、Lift,IV,True
DJK_X1667,IV、Lift,Lift,IV,IV,True
DJK_X528,IV、Lift,Lift,IV、AUC、Lift,IV,True
DK_X1315,IV、KS、AUC、Lift,KS,IV、KS、AUC,Lift,False
DKD1_X1955,IV、KS、AUC、Lift,IV、KS、AUC,IV、KS、AUC,KS,False


训练和验证、短期和长期均至少一个指标有效： 567


## 5. 第五步：PSI 稳定性

只接收第四步输出。PSI 比较训练与验证的原始字段分布（含缺失），阈值为 0.25。
`oot_df` 是接口参数名，这里传入参与开发筛选的验证集；真正时间外评估放在后面。

In [9]:
psi_selector = PSISelector(threshold=LIMITS["PSI"], oot_df=X_valid_after_effect,
                           n_jobs=N_JOBS, parallel_backend="threading")
X_after_psi = psi_selector.fit_transform(X_after_effect, y_train[LABELS[0]])
X_valid_after_psi = psi_selector.transform(X_valid_after_effect)
stable_features = list(X_after_psi)
assert stable_features, "有效字段未通过 PSI 检查"
display(psi_selector.report()[0])

,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,PSISelector,PSISelector,single,已完成,567,567,0,1.0000,6.7858,筛选完成


## 6. 第六步：相关性去冗余

只接收第五步输出。`CorrSelector` 使用训练集分箱结果的 Spearman 相关性，阈值为 **0.80**。
相互高度相关时，优先保留训练短期 IV 较高的字段；IV 在这里是排序权重，**不是额外的入选硬门槛**。
内部的分箱规则只在训练集拟合，`transform` 返回所选 X 字段的原始值。

In [10]:
corr_selector = CorrSelector(threshold=LIMITS["相关性"], method="spearman", metric="iv",
                             bin_params=BIN_PARAMS, n_jobs=N_JOBS, parallel_backend="threading")
X_selected_train = corr_selector.fit_transform(X_after_psi, y_train[LABELS[0]])
X_selected_valid = corr_selector.transform(X_valid_after_psi)
final_features = tuple(X_selected_train.columns)
frozen_features = final_features
evidence["验证PSI"] = psi_selector.scores_.reindex(evidence.index)
evidence["最终保留"] = evidence.index.isin(final_features)
selected_table = evidence.loc[list(final_features)].copy()

assert set(final_features) <= set(X.columns)
assert selected_table["双标签有效性通过"].all()
assert selected_table["验证PSI"].lt(LIMITS["PSI"]).all()
pd.testing.assert_frame_equal(X_selected_train, X_train.loc[:, list(final_features)])
display(pd.DataFrame({"阶段": ["全部X", "缺失率", "单一值率", "类型分支", "双标签任一有效", "PSI", "相关性"],
                     "字段数": [X_train.shape[1], X_after_null.shape[1], X_after_mode.shape[1], X_after_type.shape[1],
                                X_after_effect.shape[1], X_after_psi.shape[1], X_selected_train.shape[1]]}))
display(corr_selector.report()[0])
print(f"最终候选 {len(final_features)} 个：", list(final_features))

# 下文算法演示和复核复用第四步保存的训练规则，不另行拟合映射。
binner = effect_selector.binner_
B_train, B_valid = effect_selector.bins_train_, effect_selector.bins_valid_
R_train, R_valid = effect_selector.risk_train_, effect_selector.risk_valid_
Xq_train, Xq_valid = X_after_type, X_valid_after_type
iv_table = evidence[[f"训练_{label}_IV" for label in LABELS]].copy()
iv_table.columns = LABELS
iv_table["两标签最小IV"] = iv_table.min(axis=1)

def risk_transform(bins, features):
    return map_risk(bins.loc[:, list(features)], effect_selector.risk_maps_, effect_selector.base_risk_)

,阶段,字段数
0,全部X,6781
1,缺失率,6646
2,单一值率,4742
3,类型分支,4248
4,双标签任一有效,567
5,PSI,567
6,相关性,83


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,CorrSelector,CorrSelector,single,已完成,567,83,484,0.1464,7.1188,按保留优先权重完成相关冲突筛选


最终候选 83 个： ['province', 'DJK_X230', 'DJK_X345', 'DKD1_X1955', 'DKD1_X1950', 'DKD1_X1928', 'DKD1_X1051', 'DKD1_X1436', 'DK_X1293', 'DK_X2765', 'DK_X2761', 'DK_X26', 'DK_X2428', 'DK_X1453', 'DK_X1370', 'DKD1_X873', 'DKD1_X872', 'DKD1_X90', 'DKD1_X1991', 'DKD1_X185', 'DKD1_X1201', 'DK_X117', 'DKD1_X1049', 'DK_X483', 'DKD1_X404', 'DKD1_X1523', 'DKD1_X1022', 'DK_X285', 'DK_X2577', 'DK_X1592', 'DKD1_X974', 'DKD1_X67', 'DKD1_X489', 'DKD1_X1162', 'DK_X2336', 'DK_X552', 'DK_X2083', 'DKD1_X775', 'DKD1_X1784', 'DKD1_X1635', 'DKD1_X1633', 'DK_X2659', 'DK_X16', 'DK_X374', 'DK_X211', 'DK_X298', 'DK_X1799', 'DK_X655', 'DK_X2513', 'DK_X1812', 'DK_X2792', 'DK_X2584', 'DK_X100', 'DKD1_X1433', 'DK_X2536', 'DKD1_X1804', 'DKD1_X1773_2', 'BASE_X244', 'BASE_X195', 'BASE_X73', 'BASE_X291', 'BASE_X258', 'BASE_X174', 'BASE_X138', 'BASE_X147', 'BASE_X11', 'BASE_X211', 'BASE_X99', 'BASE_X115', 'BASE_X132', 'BASE_X231', 'PCO_X28', 'PCO_X13', 'DJK_X405', 'DJK_X278', 'JH_X74', 'VAR0176', 'QJVAR005', 'QJVAR071', 'NEW

## 7. 时间外复核：每个标签任一指标达标

字段集合已冻结。时间外复核使用同样的 OR 规则，不再要求四项全部达标；两种标签仍都要有效，且时间外 PSI < 0.25。
命中指标与方向反转单独展示。复核结果不回写 `final_features`；不要根据本次结果反复调参后继续把同一批样本当作全新的测试集。

In [11]:
X_oot = X.loc[split_index["时间外"], quality_features]
y_oot = Y.loc[split_index["时间外"]]
B_oot = binner.transform(X_oot)
R_oot = risk_transform(B_oot, final_features)
oot_review = pd.DataFrame(index=pd.Index(final_features, name="特征"))
oot_group_pass = pd.DataFrame(index=oot_review.index)
for label in LABELS:
    values, hits, _, _ = evaluate_metrics(B_oot.loc[:, list(final_features)], R_oot, y_oot[label], LIMITS, N_JOBS)
    oot_review = oot_review.join(values.add_prefix(label + "_"))
    oot_review[label + "_命中指标"] = hits.apply(lambda row: "、".join(row.index[row]), axis=1)
    oot_group_pass[label] = hits.any(axis=1)

oot_psi = PSISelector(threshold=LIMITS["PSI"], oot_df=X_oot.loc[:, list(final_features)],
                      n_jobs=N_JOBS, parallel_backend="threading").fit(X_after_type.loc[:, list(final_features)])
oot_review["时间外PSI"] = oot_psi.scores_
oot_review["方向反转提示"] = oot_review[[label + "_AUC" for label in LABELS]].lt(0.5).any(axis=1)
oot_review["时间外复核通过"] = oot_group_pass.all(axis=1) & oot_review["时间外PSI"].lt(LIMITS["PSI"])
oot_review["复核结论"] = [
    "；".join([label + "四项均未达标" for label in LABELS if not oot_group_pass.loc[f, label]]
             + (["PSI未达标"] if not oot_review.loc[f, "时间外PSI"] < LIMITS["PSI"] else [])) or "双标签均至少一项达标且PSI通过"
    for f in final_features
]
review_passed_features = tuple(oot_review.index[oot_review["时间外复核通过"]])
assert final_features == frozen_features
with pd.option_context("display.float_format", "{:.6f}".format):
    display(oot_review)
display(oot_review["时间外复核通过"].value_counts().rename_axis("复核通过").rename("字段数").to_frame())
print(f"时间外复核通过 {len(review_passed_features)} 个：", list(review_passed_features))

,MOB1@30_IV,MOB1@30_KS,MOB1@30_AUC,MOB1@30_Lift,MOB1@30_命中指标,MOB6@30_IV,MOB6@30_KS,MOB6@30_AUC,MOB6@30_Lift,MOB6@30_命中指标,时间外PSI,方向反转提示,时间外复核通过,复核结论
特征,,,,,,,,,,,,,,
province,0.038354,0.085649,0.450591,0.764004,IV、KS,0.003038,0.015048,0.499439,0.944661,,0.034806,True,False,MOB6@30四项均未达标
DJK_X230,0.125445,0.123596,0.573401,1.619422,IV、KS、AUC、Lift,0.017254,0.043846,0.522546,1.202563,Lift,0.001991,False,True,双标签均至少一项达标且PSI通过
DJK_X345,0.091710,0.054524,0.480125,0.493578,IV、KS,0.040793,0.029588,0.493668,0.848547,IV,0.004343,True,True,双标签均至少一项达标且PSI通过
DKD1_X1955,0.078245,0.062841,0.495665,0.581726,IV、KS,0.020812,0.044592,0.515060,0.964003,IV,0.031225,True,True,双标签均至少一项达标且PSI通过
DKD1_X1950,0.119979,0.095599,0.547287,1.040510,IV、KS、AUC,0.014440,0.043097,0.519604,1.084449,,0.030143,False,False,MOB6@30四项均未达标
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
QJVAR071,0.046977,0.066634,0.517535,0.865146,IV、KS,0.022937,0.068609,0.530816,0.985407,IV、KS、AUC,0.040029,False,True,双标签均至少一项达标且PSI通过
NEWVAR_357,0.038315,0.072077,0.533814,1.209380,IV、KS、AUC、Lift,0.010117,0.024529,0.512558,0.910051,,0.019736,False,False,MOB6@30四项均未达标
NEWVAR_529,0.039303,0.046527,0.501283,1.190494,IV,0.015293,0.042605,0.521105,1.137369,,0.057680,False,False,MOB6@30四项均未达标


,字段数
复核通过,
False,48
True,35


时间外复核通过 35 个： ['DJK_X230', 'DJK_X345', 'DKD1_X1955', 'DK_X2765', 'DK_X2761', 'DK_X26', 'DK_X1453', 'DK_X1370', 'DKD1_X873', 'DKD1_X872', 'DKD1_X185', 'DKD1_X1049', 'DKD1_X404', 'DK_X285', 'DK_X2577', 'DKD1_X67', 'DKD1_X489', 'DKD1_X1784', 'DKD1_X1635', 'DK_X2792', 'DK_X100', 'DK_X2536', 'DKD1_X1804', 'BASE_X244', 'BASE_X195', 'BASE_X73', 'BASE_X291', 'BASE_X147', 'BASE_X99', 'BASE_X132', 'BASE_X231', 'DJK_X405', 'DJK_X278', 'QJVAR071', 'XJD_VAR004']


### 多变量模型的补充检查

单变量共同有效不等于组合后一定带来增益。固定同一个逻辑回归配置，分别对两种标签比较“所有稳定候选”与“相关性去冗余后的字段”。
模型和标准化只在训练集拟合，验证/时间外只预测；这是诊断性比较，不以时间外结果挑模型。
这里输入训练坏率映射后的风险分数；`X_selected_train`、`X_selected_valid` 则保留原始 X 值，便于后续另建正式模型。

In [12]:
R_oot_stable = risk_transform(B_oot, stable_features)
model_rows = []
for scope, features in [("去冗余前", stable_features), ("最终候选", list(final_features))]:
    for label in LABELS:
        model = Pipeline([("标准化", StandardScaler()), ("逻辑回归", LogisticRegression(C=0.1, max_iter=2000, random_state=SEED))])
        model.fit(R_train[features], y_train[label])
        for split, risk, labels in [("验证", R_valid, y_valid), ("时间外", R_oot_stable, y_oot)]:
            pred = pd.DataFrame({"模型风险分": model.predict_proba(risk[features])[:, 1]}, index=risk.index)
            ks = KSSelector(n_jobs=1).fit(pred, labels[label]).scores_.iloc[0]
            model_rows.append({"字段方案": scope, "标签": label, "样本集": split, "字段数": len(features),
                               "AUC": roc_auc_score(labels[label], pred["模型风险分"]), "KS": ks})
display(pd.DataFrame(model_rows))

,字段方案,标签,样本集,字段数,AUC,KS
0,去冗余前,MOB1@30,验证,567,0.5612,0.1553
1,去冗余前,MOB1@30,时间外,567,0.4625,0.0945
2,去冗余前,MOB6@30,验证,567,0.5284,0.0751
3,去冗余前,MOB6@30,时间外,567,0.5252,0.0609
4,最终候选,MOB1@30,验证,83,0.5796,0.2301
5,最终候选,MOB1@30,时间外,83,0.4757,0.1026
6,最终候选,MOB6@30,验证,83,0.5441,0.0956
7,最终候选,MOB6@30,时间外,83,0.5300,0.0637


## 8. 更多筛选器：在小型训练候选池上演示

以下仅演示用法，不改变 `final_features`。为了使 RFE / SFS / Boruta 等反复训练的方法能在普通机器上运行，
仅取**训练双标签最小 IV 最高的 12 个质量合格字段**。所有训练行都使用，缩小的是演示字段池，主流程没有这个截断。
输入为完整、有限的分箱风险分数（取值数由箱数决定，因此 IV 不再重复分箱）。统计方法结果表达不同，不应把所有方法的交集机械地作为最终名单。

In [13]:
demo_features = iv_table.sort_values("两标签最小IV", ascending=False).head(12).index.tolist()
Xd = R_train[demo_features]
yd = y_train[LABELS[0]]
demo_k = max(1, min(5, len(demo_features) - 1))
assert len(demo_features) >= 2, "包装法演示至少需要两个候选字段"
demo_selectors = {}

def show_selection(name, selector, frame=Xd, target=yd):
    """拟合单个演示筛选器，返回对象以便继续 transform / report。"""
    selector.fit(frame, target)
    demo_selectors[name] = selector
    summary, details = selector.report()
    display(summary)
    print(name, "保留字段：", selector.selected_features_)
    return selector

forest = RandomForestClassifier(n_estimators=60, max_depth=4, min_samples_leaf=40, random_state=SEED, n_jobs=1)

### 8.1 重要性、置乱重要性与统计过滤

`FeatureImportanceSelector` 可接收 sklearn 或兼容的 hscredit 模型，读取 `feature_importances_` / `coef_`。
`NullImportanceSelector` 对比真实标签与置乱标签的归一化重要性占比，保留占比差 > 0 的字段；这里 `cv=2, n_runs=2` 仅为快速演示，正式分析应增加重复次数。
互信息捕捉非线性依赖；卡方要求非负输入；F 检验偏向组间均值差。`k` 表示保留前 k 列，并不是有效性下限。

In [14]:
importance = show_selection("模型重要性", FeatureImportanceSelector(estimator=clone(forest), threshold=1 / len(demo_features), n_jobs=N_JOBS, parallel_backend="threading"))
null_importance = show_selection("置乱重要性", NullImportanceSelector(estimator=clone(forest), threshold=0.0, cv=2, n_runs=2, random_state=SEED, n_jobs=N_JOBS, parallel_backend="threading"))
display(null_importance.get_importance_details())
mi = show_selection("互信息", MutualInfoSelector(threshold=0.005, random_state=SEED, n_jobs=N_JOBS, parallel_backend="threading"))
nonnegative = pd.DataFrame(MinMaxScaler().fit_transform(Xd), columns=Xd.columns, index=Xd.index)
chi2 = show_selection("卡方前k列", Chi2Selector(k=demo_k, n_jobs=N_JOBS, parallel_backend="threading"), frame=nonnegative)
ftest = show_selection("F检验前k列", FTestSelector(k=demo_k, n_jobs=N_JOBS, parallel_backend="threading"))

,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,FeatureImportanceSelector,FeatureImportanceSelector,single,已完成,12,5,7,0.4167,0.2012,模型重要性判定完成


模型重要性 保留字段： ['BASE_X231', 'BASE_X303', 'BASE_X244', 'QJVAR004', 'DK_X551']


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,NullImportanceSelector,NullImportanceSelector,single,已完成,12,5,7,0.4167,1.0173,完成全部实际与随机标签实验


置乱重要性 保留字段： ['BASE_X231', 'BASE_X303', 'BASE_X142', 'QJVAR004', 'DK_X551']


,特征,实际重要性,Null重要性,实际重要性%,Null重要性%,特征得分
0,BASE_X245,0.0669,0.0777,0.0669,0.0777,-0.0107
1,BASE_X231,0.1153,0.0640,0.1153,0.0640,0.0513
2,BASE_X303,0.0706,0.0686,0.0706,0.0686,0.0020
3,BASE_X15,0.0632,0.0686,0.0632,0.0686,-0.0054
4,BASE_X291_9,0.0669,0.0785,0.0669,0.0785,-0.0116
5,BASE_X244,0.0864,0.1122,0.0864,0.1122,-0.0257
6,BASE_X205,0.0611,0.0772,0.0611,0.0772,-0.0161
7,BASE_X142,0.0768,0.0734,0.0768,0.0734,0.0034
8,BASE_X256,0.0683,0.0854,0.0683,0.0854,-0.0171
9,QJVAR004,0.1197,0.1018,0.1197,0.1018,0.0179


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,MutualInfoSelector,MutualInfoSelector,single,已完成,12,0,12,0.0000,0.2664,筛选完成


互信息 保留字段： []


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,Chi2Selector,Chi2Selector,single,已完成,12,5,7,0.4167,0.0656,筛选完成


卡方前k列 保留字段： ['BASE_X245', 'BASE_X231', 'BASE_X303', 'BASE_X15', 'DK_X551']


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,FTestSelector,FTestSelector,single,已完成,12,5,7,0.4167,0.0182,筛选完成


F检验前k列 保留字段： ['BASE_X245', 'BASE_X231', 'BASE_X303', 'BASE_X15', 'BASE_X291_9']


### 8.2 RFE、SFS、逐步回归、Boruta 与 VIF

- RFE 每轮剔除较弱字段；`step` 越大计算越快，`n_features_to_select` 控制数量。
- SFS 按交叉验证 AUC 前向选择。传入的 Pipeline 将标准化放在每折内部；当前风险映射及候选池来自整个训练集，因此其内部 CV 分数仅供算法演示，不能作为无偏评估。
- Stepwise 按 AIC 等准则逐步增减字段；AIC 不是时间外效果。
- Boruta 与随机影子字段比较；20 次迭代仅演示，未确认字段不等于确定无效。
- VIF 适合检查数值设计矩阵中的多重共线性，可在相关性筛选后使用；此处不额外改动主流程结果。

In [15]:
rfe = show_selection("递归消除", RFESelector(estimator=clone(forest), n_features_to_select=demo_k, step=2, n_jobs=N_JOBS, parallel_backend="threading"))
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
sfs_model = Pipeline([("标准化", StandardScaler()), ("逻辑回归", LogisticRegression(max_iter=1000, random_state=SEED))])
sfs = show_selection("序列前向", SequentialFeatureSelector(estimator=sfs_model, n_features_to_select=demo_k, direction="forward", scoring="roc_auc", cv=cv, n_jobs=N_JOBS, parallel_backend="threading"))
scaled = pd.DataFrame(StandardScaler().fit_transform(Xd), columns=Xd.columns, index=Xd.index)
stepwise = show_selection("逐步回归", StepwiseSelector(estimator="logit", direction="forward", criterion="aic", max_features=demo_k, max_iter=10, n_jobs=1), frame=scaled)
boruta = show_selection("影子字段", BorutaSelector(estimator=clone(forest), n_estimators=60, max_iter=20, random_state=SEED, n_jobs=N_JOBS, parallel_backend="threading"))
vif = show_selection("方差膨胀因子", VIFSelector(threshold=10.0, max_iter=20, n_jobs=N_JOBS, parallel_backend="threading"), frame=scaled)

,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,RFESelector,RFESelector,single,已完成,12,5,7,0.4167,0.7611,达到包含强制保留字段的总数量预算


递归消除 保留字段： ['BASE_X245', 'BASE_X231', 'BASE_X15', 'BASE_X244', 'DK_X551']


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,SequentialFeatureSelector,SequentialFeatureSelector,single,已完成,12,5,7,0.4167,1.8490,达到包含强制保留字段的总数量预算


序列前向 保留字段： ['BASE_X231', 'BASE_X15', 'BASE_X256', 'QJVAR004', 'BASE_X275']


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,StepwiseSelector,StepwiseSelector,single,已完成,12,5,7,0.4167,1.9401,达到最大特征数


逐步回归 保留字段： ['BASE_X303', 'BASE_X15', 'BASE_X244', 'QJVAR004', 'DK_X551']


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,BorutaSelector,BorutaSelector,single,已完成,12,0,12,0.0000,3.7558,完成固定轮数并执行多重比较校正二项检验


影子字段 保留字段： []


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,VIFSelector,VIFSelector,single,已完成,12,9,3,0.7500,0.2059,所有可剔除字段满足VIF阈值


方差膨胀因子 保留字段： ['BASE_X245', 'BASE_X15', 'BASE_X291_9', 'BASE_X244', 'BASE_X205', 'BASE_X142', 'QJVAR004', 'BASE_X275', 'DK_X551']


## 9. 更多组合方式、分箱复用和双接口

以下三组使用同一个小型训练候选池：串行组合表示逐级满足；交集组合表示各条件都满足；并集组合表示任一条件满足。
并集使用本 Notebook 的 `UnionFeatureSelector`，不向内置组合器传入不支持的策略。
评分卡与稳定性感知筛选另作对照，**不将它们的额外条件叠加为主流程硬门槛**。

In [16]:
for strategy in ["sequential", "intersection"]:
    combination = CompositeFeatureSelector(selectors=[IVSelector(threshold=LIMITS["IV"], n_jobs=N_JOBS, parallel_backend="threading"), KSSelector(threshold=LIMITS["KS"], n_jobs=N_JOBS, parallel_backend="threading")], strategy=strategy, n_jobs=N_JOBS, parallel_backend="threading")
    show_selection("组合_" + strategy, combination)

union_demo = show_selection("组合_任一有效", UnionFeatureSelector([
    ("IV", IVSelector(threshold=LIMITS["IV"], n_jobs=1)),
    ("KS", KSSelector(threshold=LIMITS["KS"], n_jobs=1)),
    ("AUC", AUCSelector(threshold=LIMITS["AUC"], n_jobs=1)),
    ("Lift", LiftSelector(threshold=LIMITS["头部Lift"] - 1, direction="bad", ratio=0.1, n_jobs=1)),
], n_jobs=N_JOBS))

# 分箱器显式复用：训练一次，交给筛选器；transform 仍返回原始字段值。
demo_binner = OptimalBinning(method="quantile", max_n_bins=5, min_n_bins=1, min_bin_size=0.05, n_jobs=N_JOBS, parallel_backend="threading").fit(Xq_train[demo_features], yd)
shared_iv = IVSelector(threshold=LIMITS["IV"], binner=demo_binner, n_jobs=N_JOBS, parallel_backend="threading").fit(Xq_train[demo_features], yd)
transformed = shared_iv.transform(Xq_valid[demo_features])
pd.testing.assert_frame_equal(transformed, Xq_valid[shared_iv.selected_features_])

# df + target 接口；另一标签始终不混入输入。target_rm=True 显式移除目标列。
df_demo = Xq_train[demo_features].assign(**{LABELS[0]: yd})
target_api = IVSelector(target=LABELS[0], target_rm=True, threshold=LIMITS["IV"], binner=demo_binner, n_jobs=N_JOBS, parallel_backend="threading").fit(df_demo)
assert target_api.selected_features_ == shared_iv.selected_features_
assert LABELS[0] not in target_api.transform(df_demo)

manual = IVSelector(threshold=LIMITS["IV"], include=[demo_features[0]], force_drop=[demo_features[-1]], n_jobs=N_JOBS, parallel_backend="threading").fit(Xd, yd)
assert demo_features[0] in manual.selected_features_ and demo_features[-1] not in manual.selected_features_
display(manual.report()[1].query("记录类型 == '特征决策'")[["特征", "决策来源", "筛选结果", "筛选原因"]])

,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,CompositeFeatureSelector,CompositeFeatureSelector,sequential,已完成,12,12,0,1.0000,0.0944,筛选完成
1,2,selector/01_IVSelector,selector,IVSelector,IVSelector,single,已完成,12,12,0,1.0000,0.0328,筛选完成
2,3,selector/02_KSSelector,selector,KSSelector,KSSelector,single,已完成,12,12,0,1.0000,0.0380,筛选完成


组合_sequential 保留字段： ['BASE_X245', 'BASE_X231', 'BASE_X303', 'BASE_X15', 'BASE_X291_9', 'BASE_X244', 'BASE_X205', 'BASE_X142', 'BASE_X256', 'QJVAR004', 'BASE_X275', 'DK_X551']


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,CompositeFeatureSelector,CompositeFeatureSelector,intersection,已完成,12,12,0,1.0000,0.0951,筛选完成
1,2,selector/01_IVSelector,selector,IVSelector,IVSelector,single,已完成,12,12,0,1.0000,0.0342,筛选完成
2,3,selector/02_KSSelector,selector,KSSelector,KSSelector,single,已完成,12,12,0,1.0000,0.0627,筛选完成


组合_intersection 保留字段： ['BASE_X245', 'BASE_X231', 'BASE_X303', 'BASE_X15', 'BASE_X291_9', 'BASE_X244', 'BASE_X205', 'BASE_X142', 'BASE_X256', 'QJVAR004', 'BASE_X275', 'DK_X551']


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,UnionFeatureSelector,UnionFeatureSelector,union,已完成,12,12,0,1.0000,0.1234,筛选完成
1,2,selector/01_IV,selector,IV,IVSelector,single,已完成,12,12,0,1.0000,0.0142,筛选完成
2,3,selector/02_KS,selector,KS,KSSelector,single,已完成,12,12,0,1.0000,0.0143,筛选完成
3,4,selector/03_AUC,selector,AUC,AUCSelector,single,已完成,12,12,0,1.0000,0.0289,筛选完成
4,5,selector/04_Lift,selector,Lift,LiftSelector,single,已完成,12,12,0,1.0000,0.0162,筛选完成


组合_任一有效 保留字段： ['BASE_X245', 'BASE_X231', 'BASE_X303', 'BASE_X15', 'BASE_X291_9', 'BASE_X244', 'BASE_X205', 'BASE_X142', 'BASE_X256', 'QJVAR004', 'BASE_X275', 'DK_X551']


,特征,决策来源,筛选结果,筛选原因
0,BASE_X245,强制保留,保留,用户强制保留
1,BASE_X231,算法,保留,满足筛选条件
2,BASE_X303,算法,保留,满足筛选条件
3,BASE_X15,算法,保留,满足筛选条件
4,BASE_X291_9,算法,保留,满足筛选条件
5,BASE_X244,算法,保留,满足筛选条件
6,BASE_X205,算法,保留,满足筛选条件
7,BASE_X142,算法,保留,满足筛选条件
8,BASE_X256,算法,保留,满足筛选条件
9,QJVAR004,算法,保留,满足筛选条件


In [17]:
# 自动分箱参数接口；评分卡顺序为缺失率 → IV → 相关性 → 单一值率。
scorecard = show_selection("评分卡一站式", ScorecardFeatureSelection(
    null_threshold=LIMITS["缺失率"], iv_threshold=LIMITS["IV"], corr_threshold=LIMITS["相关性"], mode_threshold=LIMITS["单一值率"],
    bin_params={"method": "quantile", "max_n_bins": 5, "min_n_bins": 1, "min_bin_size": 0.05}, n_jobs=N_JOBS, parallel_backend="threading"), frame=Xq_train[demo_features])

stability = show_selection("稳定性感知", StabilityAwareSelector(iv_threshold=LIMITS["IV"], psi_threshold=LIMITS["PSI"],
    oot_df=Xq_valid[demo_features], binner=demo_binner, random_state=SEED, n_jobs=N_JOBS, parallel_backend="threading"), frame=Xq_train[demo_features])
display(stability.get_detail())

pipeline = Pipeline([
    ("方差筛选", VarianceSelector(threshold=0, n_jobs=1)),
    ("标准化", StandardScaler()),
    ("逻辑回归", LogisticRegression(max_iter=2000, random_state=SEED)),
])
pipeline.fit(Xd, yd)
print("Pipeline 验证 AUC：", round(roc_auc_score(y_valid[LABELS[0]], pipeline.predict_proba(R_valid[demo_features])[:, 1]), 4))
display(pd.DataFrame([{"演示方法": name, "保留字段数": len(selector.selected_features_), "保留字段": selector.selected_features_} for name, selector in demo_selectors.items()]))

,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,ScorecardFeatureSelection,ScorecardFeatureSelection,sequential,已完成,12,5,7,0.4167,0.2822,筛选完成
1,2,selector/01_缺失率筛选,selector,缺失率筛选,NullSelector,single,已完成,12,12,0,1.0000,0.0028,筛选完成
2,3,selector/02_IV值筛选,selector,IV值筛选,IVSelector,single,已完成,12,12,0,1.0000,0.0336,筛选完成
3,4,selector/03_相关性筛选,selector,相关性筛选,CorrSelector,single,已完成,12,5,7,0.4167,0.0526,按保留优先权重完成相关冲突筛选
4,5,selector/04_单一值筛选,selector,单一值筛选,ModeSelector,single,已完成,5,5,0,1.0000,0.0165,筛选完成


评分卡一站式 保留字段： ['BASE_X15', 'BASE_X244', 'BASE_X142', 'QJVAR004', 'DK_X551']


,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,selector,None,StabilityAwareSelector,StabilityAwareSelector,single,已完成,12,12,0,1.0000,0.1294,筛选完成


稳定性感知 保留字段： ['BASE_X245', 'BASE_X231', 'BASE_X303', 'BASE_X15', 'BASE_X291_9', 'BASE_X244', 'BASE_X205', 'BASE_X142', 'BASE_X256', 'QJVAR004', 'BASE_X275', 'DK_X551']


,IV,PSI,综合评分,入选
BASE_X15,0.2195,0.0211,0.3151,True
QJVAR004,0.1647,0.0145,0.2552,True
BASE_X245,0.1931,0.0222,0.2280,True
BASE_X303,0.1858,0.0211,0.2228,True
BASE_X231,0.1969,0.0290,0.1472,True
BASE_X142,0.1330,0.0176,0.1258,True
BASE_X291_9,0.1830,0.0297,0.1002,True
BASE_X205,0.1436,0.0223,0.0921,True
BASE_X244,0.1524,0.0242,0.0897,True
BASE_X275,0.1430,0.0238,0.0698,True


Pipeline 验证 AUC： 0.6127


,演示方法,保留字段数,保留字段
0,模型重要性,5,"[BASE_X231, BASE_X303, BASE_X244, QJVAR004, DK_X551]"
1,置乱重要性,5,"[BASE_X231, BASE_X303, BASE_X142, QJVAR004, DK_X551]"
2,互信息,0,[]
3,卡方前k列,5,"[BASE_X245, BASE_X231, BASE_X303, BASE_X15, DK_X551]"
4,F检验前k列,5,"[BASE_X245, BASE_X231, BASE_X303, BASE_X15, BASE_X291_9]"
5,递归消除,5,"[BASE_X245, BASE_X231, BASE_X15, BASE_X244, DK_X551]"
6,序列前向,5,"[BASE_X231, BASE_X15, BASE_X256, QJVAR004, BASE_X275]"
7,逐步回归,5,"[BASE_X303, BASE_X15, BASE_X244, QJVAR004, DK_X551]"
8,影子字段,0,[]
9,方差膨胀因子,9,"[BASE_X245, BASE_X15, BASE_X291_9, BASE_X244, BASE_X205, BASE_X142, QJVAR004, BASE_X275, DK_X551]"


## 10. 汇总独立筛选器与组合分支的报告

所有原始 X 字段都有去向；未进入后续阶段的指标保留缺失值，不伪造为零。
明细中的“命中指标”解释任一有效规则；每个指标的原始数值保留在 `evidence` 与 `final_review`。

In [18]:
collector = SelectionReportCollector(name="双标签任一有效特征筛选", relation="independent")
for name, step in [("缺失率", null_selector), ("单一值率", mode_selector), ("类型分支组合", type_selector)]:
    collector.add_source(step, stage_name=name)
for group, (iv, risk_union) in effect_selector.sources_.items():
    collector.add_source(iv, stage_name=group + "_IV")
    collector.add_source(risk_union, stage_name=group + "_风险指标并集")
collector.add_source(psi_selector, stage_name="验证PSI")
collector.add_source(corr_selector, stage_name="相关性去冗余")
report_summary, report_process = collector.report()
display(report_summary)

feature_audit = pd.DataFrame(index=pd.Index(X.columns, name="特征"))
feature_audit["质量合格"] = feature_audit.index.isin(quality_features)
feature_audit["双标签任一有效"] = feature_audit.index.isin(performance_features)
feature_audit["验证PSI通过"] = feature_audit.index.isin(stable_features)
feature_audit["最终保留"] = feature_audit.index.isin(final_features)
feature_audit["去向"] = np.select(
    [~feature_audit["质量合格"], ~feature_audit["双标签任一有效"], ~feature_audit["验证PSI通过"], ~feature_audit["最终保留"]],
    ["质量筛选剔除", "至少一组四项均未达标", "分布漂移", "相关性去冗余"], default="保留")
feature_audit = feature_audit.join(evidence.drop(columns="最终保留"))
display(feature_audit.groupby("去向", sort=False).size().rename("字段数").to_frame())

final_review = selected_table.join(oot_review.add_prefix("时间外_"))
assert feature_audit.index.equals(pd.Index(X.columns, name="特征"))
assert set(feature_audit.index[feature_audit["最终保留"]]) == set(final_features)
assert final_features == frozen_features
display(final_review[[c for c in final_review if "命中指标" in c] + ["验证PSI", "时间外_时间外PSI", "时间外_时间外复核通过"]])

,阶段编号,阶段路径,父路径,阶段名称,筛选器,关系类型,执行状态,输入特征数,选中特征数,剔除特征数,保留率,耗时秒,停止原因
0,1,list/01_缺失率,None,缺失率,NullSelector,single,已完成,6781,6646,135,0.9801,0.5224,筛选完成
1,2,list/02_单一值率,None,单一值率,ModeSelector,single,已完成,6646,4742,1904,0.7135,2.6682,筛选完成
2,3,list/03_类型分支组合,None,类型分支组合,UnionFeatureSelector,union,已完成,4742,4248,494,0.8958,4.5678,筛选完成
3,4,list/03_类型分支组合/01_数值分支,list/03_类型分支组合,数值分支,CompositeFeatureSelector,sequential,已完成,4742,4238,504,0.8937,2.1264,筛选完成
4,5,list/03_类型分支组合/01_数值分支/01_数值类型,list/03_类型分支组合/01_数值分支,数值类型,TypeSelector,single,已完成,4742,4732,10,0.9979,0.4008,筛选完成
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26,27,list/11_验证_MOB6@30_风险指标并集/01_KS,list/11_验证_MOB6@30_风险指标并集,KS,KSSelector,single,已完成,4248,608,3640,0.1431,3.5489,筛选完成
27,28,list/11_验证_MOB6@30_风险指标并集/02_AUC,list/11_验证_MOB6@30_风险指标并集,AUC,AUCSelector,single,已完成,4248,174,4074,0.0410,4.2450,筛选完成
28,29,list/11_验证_MOB6@30_风险指标并集/03_Lift,list/11_验证_MOB6@30_风险指标并集,Lift,LiftSelector,single,已完成,4248,557,3691,0.1311,1.5063,筛选完成
29,30,list/12_验证PSI,None,验证PSI,PSISelector,single,已完成,567,567,0,1.0000,6.7858,筛选完成


,字段数
去向,
保留,83
至少一组四项均未达标,3681
质量筛选剔除,2533
相关性去冗余,484


,训练_MOB1@30_命中指标,训练_MOB6@30_命中指标,验证_MOB1@30_命中指标,验证_MOB6@30_命中指标,时间外_MOB1@30_命中指标,时间外_MOB6@30_命中指标,验证PSI,时间外_时间外PSI,时间外_时间外复核通过
特征,,,,,,,,,
province,IV、KS、AUC、Lift,IV、KS、AUC、Lift,IV、KS、Lift,KS,IV、KS,,0.0096,0.0348,False
DJK_X230,IV、KS、AUC、Lift,KS,IV、KS,IV、KS,IV、KS、AUC、Lift,Lift,0.0048,0.0020,True
DJK_X345,IV、AUC、Lift,Lift,IV、KS、Lift,IV,IV、KS,IV,0.0029,0.0043,True
DKD1_X1955,IV、KS、AUC、Lift,IV、KS、AUC,IV、KS、AUC,KS,IV、KS,IV,0.0128,0.0312,True
DKD1_X1950,IV、KS、AUC、Lift,KS、AUC,IV、KS、AUC,Lift,IV、KS、AUC,,0.0138,0.0301,False
...,...,...,...,...,...,...,...,...,...
QJVAR071,IV、KS、AUC、Lift,KS、AUC、Lift,IV、KS,IV,IV、KS,IV、KS、AUC,0.0173,0.0400,True
NEWVAR_357,IV、KS、AUC、Lift,IV、KS、AUC,IV、KS,IV,IV、KS、AUC、Lift,,0.0084,0.0197,False
NEWVAR_529,IV、KS、AUC、Lift,IV、KS、AUC、Lift,IV、KS、AUC,Lift,IV,,0.0229,0.0577,False


## 11. 用 Pipeline 串起相同流程，并与逐步调用核对

上面每个筛选器都可以独立运行。现在将同一组配置克隆为新对象，按完全相同的顺序交给 Pipeline：

`缺失率 → 单一值率 → 类型分支组合 → 双标签任一有效 → PSI → 相关性`

Pipeline 中每一步自动接收前一步输出。短期标签使用标准 `y` 参数；长期标签和验证样本通过 `双标签有效__...` 仅传给需要它们的步骤。
验证样本可传完整 X，该步骤内部只读取上游保留字段。PSI 同样按当前输入字段对齐。
Pipeline 不使用时间外标签。重新拟合后验证字段顺序、原始 X 值及四组任一有效的判定都与逐步调用一致。

In [19]:
selection_pipeline = Pipeline([
    ("缺失率", clone(null_selector)),
    ("单一值率", clone(mode_selector)),
    ("类型分支", clone(type_selector)),
    ("双标签有效", clone(effect_selector)),
    ("PSI", clone(psi_selector).set_params(oot_df=X_valid)),
    ("相关性", clone(corr_selector)),
])
X_pipeline_train = selection_pipeline.fit_transform(
    X_train, y_train[LABELS[0]],
    双标签有效__long_y=y_train[LABELS[1]],
    双标签有效__validation_X=X_valid,
    双标签有效__validation_labels=y_valid,
)
X_pipeline_valid = selection_pipeline.transform(X_valid)

pd.testing.assert_frame_equal(X_pipeline_train, X_selected_train)
pd.testing.assert_frame_equal(X_pipeline_valid, X_selected_valid)
pd.testing.assert_frame_equal(selection_pipeline.named_steps["双标签有效"].group_pass_, effect_selector.group_pass_)
assert list(X_pipeline_train) == list(final_features)

# 语义核对：确实有不满足四项全通过、但满足任一有效的新候选。
strict_pass = pd.Series(True, index=evidence.index)
for split in ["训练", "验证"]:
    for label in LABELS:
        for metric, threshold in METRIC_LIMITS.items():
            strict_pass &= evidence[f"{split}_{label}_{metric}"].ge(threshold)
rescued = evidence.index[evidence["双标签有效性通过"] & ~strict_pass]
assert len(rescued) > 0, "任一有效规则未产生与严格交集不同的候选，请核查实现"
print("Pipeline 与逐步执行的字段、顺序及数据值完全一致。")
print(f"全部指标严格交集 {int(strict_pass.sum())} 列；任一指标有效 {len(performance_features)} 列；新增候选 {len(rescued)} 列。")
print(f"去冗余后 {len(final_features)} 列；时间外复核通过 {len(review_passed_features)} 列。")

Pipeline 与逐步执行的字段、顺序及数据值完全一致。
全部指标严格交集 40 列；任一指标有效 567 列；新增候选 527 列。
去冗余后 83 列；时间外复核通过 35 列。


运行后可继续使用：

| 对象 | 用途 |
| --- | --- |
| `selection_pipeline` | 可复用的完整串行筛选流程；新数据调用 `transform` |
| `final_features` | 开发阶段确定并冻结的字段名，保持原始 X 顺序 |
| `X_selected_train` / `X_selected_valid` | 筛选后的 X；标签仍单独保存 |
| `effect_selector.metric_hits_` | 每组每个指标的达标矩阵，可核查 OR 判定 |
| `selected_table` / `final_review` | 候选字段的命中理由和完整数值证据 |
| `review_passed_features` / `oot_review` | 时间外复核名单及结果，不回写开发阶段字段 |
| `feature_audit` | 全部 X 的去向 |
| `collector` | 包含并列子筛选器的报告；需要归档时显式调用 `collector.get_report().save(输出路径)` |

阈值统一在 `LIMITS` 修改；修改后从头完整运行。单变量任一有效代表候选信号，不保证单变量方向、组合模型增益或每个客群效果；相关提示与时间外结果均保留，供后续建模判断。